<a href="https://colab.research.google.com/github/jihanbakshi7/Explainable-Multi-modal-Fact-Verification-System-Using-Context-Aware-Retrieval-Augmented-Evi.-Gen./blob/main/notebooks/02_retrieval_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTEBOOK 2 / 6 — Retrieval Index (fresh build, automatic downloads only)
**Runtime: GPU (T4) recommended.**

**Requires:** Notebook 1's outputs (`chunks_df.parquet`, `test_df.parquet`)
in the shared Drive checkpoint folder.

This version downloads models automatically (ModelScope first for BAAI
models, standard `huggingface_hub`/`SentenceTransformer` otherwise) — no
custom manual byte-streaming downloader. Relies on three environment fixes
established earlier, which should be enough on their own now:
1. **HF cache on local disk**, never Drive (Drive's FUSE mount breaks HF's
   symlink-based cache).
2. **A real HF token**, set via Secrets/manual paste — biggest lever against
   rate-limit stalls.
3. **`HF_XET_HIGH_PERFORMANCE=1`** — the current performance flag for
   `huggingface_hub`'s newer Xet transfer backend.

If a download still stalls with all of this in place, that's a genuine
external (HF-side) issue worth reporting/waiting out, not a code problem.


In [5]:
# ===== Cell 1: Install — GPU cell =====
!pip -q install -U pip
!pip -q install sentence-transformers faiss-cpu rank_bm25 pandas numpy tqdm pyarrow huggingface_hub modelscope
print('Dependencies installed.')


Dependencies installed.


In [6]:
# ===== Cell 2: Mount Drive + checkpoint folder — CPU =====
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_NAME = 'averitec_20claim_v1'
PROJECT_DIR = Path('/content/drive/MyDrive/averitec_extended')
CHECKPOINT_DIR = PROJECT_DIR / RUN_NAME / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_NAME)
print('Checkpoint folder:', CHECKPOINT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run: averitec_20claim_v1
Checkpoint folder: /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints


## Local Hugging Face cache + authentication
The cache is configured before importing `huggingface_hub`. Colab Secrets
is preferred; the hidden prompt is a safe one-off fallback.

In [7]:
# ===== Cell 3: Local HF cache + authentication =====
import os, getpass

# Configure the cache before importing huggingface_hub or model libraries.
HF_CACHE_DIR = Path('/content/hf_cache')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
print('HF model cache (local disk):', HF_CACHE_DIR)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print('Using token from Colab Secrets.')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('No token found. Paste your HF token now (hidden), or press Enter to skip: ').strip()

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF token active.')
else:
    print('No HF token set -- recommended: set one via Colab Secrets (key icon, left sidebar).')


HF model cache (local disk): /content/hf_cache


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Using token from Colab Secrets.
HF token active.


## Verify local cache configuration

In [8]:
# ===== Cell 4: Verify HF cache setup =====
import time

assert os.environ['HF_HOME'] == str(HF_CACHE_DIR)
assert os.environ['HF_HUB_CACHE'] == str(HF_CACHE_DIR / 'hub')
print('HF cache configuration verified.')


HF cache configuration verified.


## Model choice
Change `EMBEDDING_MODEL_CHOICE` to switch models -- everything downstream
adapts automatically.

| Key | Model | Size | Language |
|---|---|---|---|
| `bge_m3` | BAAI/bge-m3 | 2.27 GB | Multilingual |
| `bge_large_en` | BAAI/bge-large-en-v1.5 | 1.34 GB | English |
| `bge_base_en` | BAAI/bge-base-en-v1.5 | 438 MB | English -- good balance |
| `bge_small_en` | BAAI/bge-small-en-v1.5 | 133 MB | English -- fastest |
| `minilm` | sentence-transformers/all-MiniLM-L6-v2 | 90 MB | English, smallest |


In [9]:
# ===== Cell 5: Model + retrieval settings =====

EMBEDDING_MODEL_CHOICE = 'bge_large_en'

EMBEDDING_MODEL_OPTIONS = {
    'bge_m3'       : 'BAAI/bge-m3',
    'bge_large_en' : 'BAAI/bge-large-en-v1.5',
    'bge_base_en'  : 'BAAI/bge-base-en-v1.5',
    'bge_small_en' : 'BAAI/bge-small-en-v1.5',
    'minilm'       : 'sentence-transformers/all-MiniLM-L6-v2',
}
BGE_MODEL_NAME = EMBEDDING_MODEL_OPTIONS[EMBEDDING_MODEL_CHOICE]
RERANKER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
BGE_QUERY_PREFIX = (
    'Represent this sentence for searching relevant passages: '
    if EMBEDDING_MODEL_CHOICE in {'bge_large_en', 'bge_base_en', 'bge_small_en'}
    else ''
)

BM25_TOP_N, BGE_TOP_N, FINAL_TOP_K = 50, 50, 20
BM25_WEIGHT, BGE_WEIGHT = 0.50, 0.50
RERANK_BATCH_SIZE, RERANK_MAX_LEN, TOP_K_RERANKED = 16, 512, 5

print('Embedding model:', BGE_MODEL_NAME)
print('Reranker model :', RERANKER_MODEL_NAME)


Embedding model: BAAI/bge-large-en-v1.5
Reranker model : cross-encoder/ms-marco-MiniLM-L-6-v2


## Automatic download: ModelScope first, else standard huggingface_hub/SentenceTransformer
No manual byte-streaming here -- just the libraries' own downloaders, with
a simple retry-on-exception (not a custom timeout interrupt).

In [10]:
# ===== Cell 6: Automatic model resolution (ModelScope or standard HF) =====
import time

MODEL_LOCAL_PATHS = {}   # repo_id -> local folder, only set if ModelScope succeeds
MODELSCOPE_CACHE_DIR = Path('/content/modelscope_cache')
MODELSCOPE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def try_modelscope_download(repo_id):
    """ModelScope hosts BAAI's own models natively -- a different server
    than Hugging Face, worth trying automatically for any BAAI/ repo."""
    if not repo_id.startswith('BAAI/'):
        return None
    try:
        from modelscope import snapshot_download as ms_snapshot_download
        print(f'  Trying ModelScope for {repo_id}...')
        t0 = time.time()
        local_path = ms_snapshot_download(model_id=repo_id, cache_dir=str(MODELSCOPE_CACHE_DIR))
        print(f'  ModelScope succeeded in {time.time()-t0:.0f}s: {local_path}')
        return local_path
    except Exception as ex:
        print(f'  ModelScope not used ({ex}) -- will use standard Hugging Face download instead.')
        return None


for repo_id in [BGE_MODEL_NAME, RERANKER_MODEL_NAME]:
    print(f'\n--- {repo_id} ---')
    ms_path = try_modelscope_download(repo_id)
    if ms_path:
        MODEL_LOCAL_PATHS[repo_id] = ms_path
    else:
        print(f'  Will load {repo_id} via standard automatic download in the loading cells below.')

print('\nResolved via ModelScope:', list(MODEL_LOCAL_PATHS.keys()) or '(none -- using standard HF download for all)')



--- BAAI/bge-large-en-v1.5 ---
  Trying ModelScope for BAAI/bge-large-en-v1.5...


2026-08-11 20:56:51,109 | INFO    | modelscope_hub.download | Downloading 15 files from BAAI/bge-large-en-v1.5@master


Downloading:   0%|          | 0/15 [00:00<?, ?file/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

  ModelScope succeeded in 106s: /content/modelscope_cache/models/BAAI--bge-large-en-v1.5/snapshots/master

--- cross-encoder/ms-marco-MiniLM-L-6-v2 ---
  Will load cross-encoder/ms-marco-MiniLM-L-6-v2 via standard automatic download in the loading cells below.

Resolved via ModelScope: ['BAAI/bge-large-en-v1.5']


## Imports + GPU check

In [11]:
# ===== Cell 7: Imports + GPU check =====
import re, math, json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from tqdm.auto import tqdm

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n*** No GPU. Runtime -> Change runtime type -> T4 GPU for faster embedding. ***\n')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tokenize_bm25(text):
    return TOKEN_RE.findall(str(text).lower())

def minmax_normalize(values):
    values = np.asarray(values, dtype='float32')
    if values.size == 0:
        return values
    v_min, v_max = float(np.min(values)), float(np.max(values))
    if math.isclose(v_min, v_max):
        return np.zeros_like(values, dtype='float32')
    return (values - v_min) / (v_max - v_min)

print('Settings ready.')


CUDA available: True
GPU: Tesla T4
Settings ready.


In [12]:
# ===== Cell 8: Load Notebook 1's outputs — CPU =====

required_files = ['chunks_df.parquet', 'test_df.parquet']
missing = [f for f in required_files if not (CHECKPOINT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files from Notebook 1: {missing}. Run 01_data_prep.ipynb first.')

chunks_df = pd.read_parquet(CHECKPOINT_DIR / 'chunks_df.parquet')
test_df   = pd.read_parquet(CHECKPOINT_DIR / 'test_df.parquet')
print(f'Loaded chunks_df: {len(chunks_df)} chunks')
print(f'Loaded test_df  : {len(test_df)} claims')

if len(chunks_df) == 0:
    raise ValueError('chunks_df is empty -- Notebook 1 scraped 0 usable evidence documents.')


Loaded chunks_df: 634 chunks
Loaded test_df  : 20 claims


## Build BM25 index — CPU

In [13]:
# ===== Cell 9: BM25 index =====
print('Building BM25 index...')
tokenized_corpus = [tokenize_bm25(t) for t in tqdm(chunks_df['retrieval_text'].tolist(), desc='Tokenizing')]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 index ready. Corpus size: {len(tokenized_corpus)} chunks')


Building BM25 index...


Tokenizing:   0%|          | 0/634 [00:00<?, ?it/s]

BM25 index ready. Corpus size: 634 chunks


## Build dense index — automatic download, GPU strongly recommended

In [14]:
# ===== Cell 10: Dense embeddings + FAISS =====
import signal

class _LoadTimeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise _LoadTimeout()


def load_with_timeout_retry(load_fn, repo_id, source_desc, max_attempts=4, attempt_timeout_sec=900):
    """Calls load_fn() (still the SAME automatic SentenceTransformer/
    CrossEncoder call every time -- no manual downloading). If a load
    remains unfinished after attempt_timeout_sec, it is interrupted and
    retried. The longer allowance avoids interrupting a healthy first
    download. huggingface_hub resumes partial
    downloads automatically across separate calls, so no progress is lost,
    it just stops hanging forever."""
    for attempt in range(1, max_attempts + 1):
        print(f'Loading {repo_id} ({source_desc}), attempt {attempt}/{max_attempts} '
              f'(will interrupt + retry if unfinished after {attempt_timeout_sec}s)...')
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(attempt_timeout_sec)
        try:
            model = load_fn()
            signal.alarm(0)
            return model
        except _LoadTimeout:
            print(f'  Load unfinished after {attempt_timeout_sec}s -- interrupting and retrying '
                  f'(this resumes from where it left off, doesn\'t restart from scratch)...')
        except Exception as ex:
            print(f'  Attempt {attempt} failed: {ex}')
        finally:
            signal.alarm(0)
        if attempt < max_attempts:
            time.sleep(5)
    raise RuntimeError(f'Could not load {repo_id} after {max_attempts} attempts, each given '
                       f'{attempt_timeout_sec}s. This suggests a genuine, sustained HF-side '
                       f'issue right now -- try again later, or switch EMBEDDING_MODEL_CHOICE '
                       f'to a smaller model in Cell 5.')


def load_embedding_model(repo_id, max_attempts=4, attempt_timeout_sec=900):
    load_source = MODEL_LOCAL_PATHS.get(repo_id, repo_id)
    source_desc = 'local ModelScope copy' if repo_id in MODEL_LOCAL_PATHS else 'automatic HF download'
    return load_with_timeout_retry(
        lambda: SentenceTransformer(load_source, device=DEVICE),
        repo_id, source_desc, max_attempts, attempt_timeout_sec,
    )


bge_model = load_embedding_model(BGE_MODEL_NAME)
print('Loaded.')

print(f'Encoding {len(chunks_df)} chunks on {DEVICE}...')
chunk_embeddings = bge_model.encode(
    chunks_df['retrieval_text'].tolist(), batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype('float32')

embedding_dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(chunk_embeddings)
print(f'FAISS index ready. Size: {faiss_index.ntotal}, Dim: {embedding_dim}')


Loading BAAI/bge-large-en-v1.5 (local ModelScope copy), attempt 1/4 (will interrupt + retry if unfinished after 900s)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded.
Encoding 634 chunks on cuda...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

FAISS index ready. Size: 634, Dim: 1024


## Hybrid retrieval + reranker

In [15]:
# ===== Cell 11: hybrid_retrieve + rerank_chunks =====

def hybrid_retrieve(query_text, top_k=FINAL_TOP_K):
    tokenized_q = tokenize_bm25(query_text)
    bm25_scores_all = bm25.get_scores(tokenized_q)
    bm25_top_idx = np.argsort(bm25_scores_all)[::-1][:BM25_TOP_N]

    dense_query = BGE_QUERY_PREFIX + query_text
    q_emb = bge_model.encode([dense_query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    dense_scores, dense_idx = faiss_index.search(q_emb, BGE_TOP_N)
    dense_top_idx = dense_idx[0][dense_idx[0] >= 0].astype(int)

    union_idx = np.array(sorted(set(bm25_top_idx.tolist()) | set(dense_top_idx.tolist())), dtype=int)
    if len(union_idx) == 0:
        return []

    bm25_scores = bm25_scores_all[union_idx].astype('float32')
    bge_scores  = np.dot(chunk_embeddings[union_idx], q_emb[0]).astype('float32')
    combined    = BM25_WEIGHT * minmax_normalize(bm25_scores) + BGE_WEIGHT * minmax_normalize(bge_scores)

    order    = np.argsort(combined)[::-1]
    selected = order[:min(top_k, len(order))]

    results = []
    for rank_pos, local_pos in enumerate(selected, start=1):
        chunk_idx = int(union_idx[local_pos])
        row       = chunks_df.iloc[chunk_idx]
        results.append({
            'rank'          : rank_pos, 'chunk_id': row['chunk_id'], 'doc_id': row['doc_id'],
            'chunk_text'    : row['chunk_text'], 'url': row.get('url', ''),
            'combined_score': float(combined[local_pos]),
        })
    return results


def load_reranker(repo_id, max_attempts=4, attempt_timeout_sec=900):
    load_source = MODEL_LOCAL_PATHS.get(repo_id, repo_id)
    return load_with_timeout_retry(
        lambda: CrossEncoder(load_source, device=DEVICE, max_length=RERANK_MAX_LEN),
        repo_id, 'automatic HF download', max_attempts, attempt_timeout_sec,
    )


reranker = load_reranker(RERANKER_MODEL_NAME)
print('Loaded.')


def rerank_chunks(query_text, chunks, top_k=TOP_K_RERANKED):
    if not chunks:
        return []
    pairs = [(query_text, c['chunk_text']) for c in chunks]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False, convert_to_numpy=True)
    order = np.argsort(scores)[::-1]
    reranked = []
    for new_rank, idx in enumerate(order[:top_k], start=1):
        c = dict(chunks[idx]); c['rerank_rank'] = new_rank; c['rerank_score'] = float(scores[idx])
        reranked.append(c)
    return reranked


sample_claim = test_df['claim'].iloc[0]
test_results = hybrid_retrieve(sample_claim, top_k=5)
reranked_test = rerank_chunks(sample_claim, test_results)
print('\nSmoke test:')
print(f'  Query    : {sample_claim[:80]}...')
print(f'  Retrieved: {len(test_results)} -> Reranked: {len(reranked_test)}')


Loading cross-encoder/ms-marco-MiniLM-L-6-v2 (automatic HF download), attempt 1/4 (will interrupt + retry if unfinished after 900s)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded.

Smoke test:
  Query    : John Cammo was the only one to predict that President Trump would be infected wi...
  Retrieved: 5 -> Reranked: 5


## Save checkpoints for downstream notebooks

In [16]:
# ===== Cell 12: Save embeddings + FAISS index =====

np.save(CHECKPOINT_DIR / 'chunk_embeddings.npy', chunk_embeddings)
faiss.write_index(faiss_index, str(CHECKPOINT_DIR / 'faiss_index.bin'))

retrieval_config = {
    'BM25_TOP_N': BM25_TOP_N, 'BGE_TOP_N': BGE_TOP_N, 'FINAL_TOP_K': FINAL_TOP_K,
    'BM25_WEIGHT': BM25_WEIGHT, 'BGE_WEIGHT': BGE_WEIGHT,
    'BGE_MODEL_NAME': BGE_MODEL_NAME, 'RERANKER_MODEL_NAME': RERANKER_MODEL_NAME,
    'BGE_QUERY_PREFIX': BGE_QUERY_PREFIX,
    'TOP_K_RERANKED': TOP_K_RERANKED, 'embedding_dim': int(embedding_dim),
    'n_chunks': len(chunks_df), 'embedding_model_choice': EMBEDDING_MODEL_CHOICE,
}
with open(CHECKPOINT_DIR / 'retrieval_config.json', 'w') as f:
    json.dump(retrieval_config, f, indent=2)

print('Saved to', CHECKPOINT_DIR, ':')
print('  - chunk_embeddings.npy  (', chunk_embeddings.shape, ')')
print('  - faiss_index.bin       (', faiss_index.ntotal, 'vectors )')
print('  - retrieval_config.json (embedding model:', BGE_MODEL_NAME, ')')
print()
print('DONE. Open Notebook 3 (Baseline Pipeline) next.')


Saved to /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints :
  - chunk_embeddings.npy  ( (634, 1024) )
  - faiss_index.bin       ( 634 vectors )
  - retrieval_config.json (embedding model: BAAI/bge-large-en-v1.5 )

DONE. Open Notebook 3 (Baseline Pipeline) next.
